In [3]:
import subprocess, json, time, sys
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import imageio.v2 as imageio
from pathlib import Path
from tqdm import tqdm
from PIL import Image as PILImage
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from skimage.metrics import structural_similarity as ssim_fn

COLMAP_BIN = Path(r"C:\Users\e3545\Downloads\colmap-x64-windows-cuda\COLMAP.bat")

SCALE         = 0.5
device        = "cuda" if torch.cuda.is_available() else "cpu"
POS_FREQS, DIR_FREQS = 10, 4
pos_dim       = 3 + 3 * 2 * POS_FREQS
dir_dim       = 3 + 3 * 2 * DIR_FREQS
N_ITERS       = 15000
N_SAMPLES     = 64
BATCH_RAY     = 1024
PHOTO_DIR     = Path("photos/real")
COLMAP_OUT    = Path("colmap_out")

NEAR = 0.01
FAR  = 5.0

In [4]:
def run_cmd(cmd):
    # Windows 上 .bat 需要 shell=True
    is_bat = str(cmd[0]).endswith(".bat")
    r = subprocess.run(
        cmd, capture_output=True,
        encoding=sys.stdout.encoding or "utf-8", errors="replace",
        shell=is_bat,
    )
    if r.returncode != 0:
        raise RuntimeError(f"COLMAP failed at '{cmd[1]}':\n{r.stderr[-400:]}")

def quat_to_rot(qw, qx, qy, qz):
    return np.array([
        [1-2*(qy**2+qz**2),   2*(qx*qy-qz*qw),   2*(qx*qz+qy*qw)],
        [  2*(qx*qy+qz*qw), 1-2*(qx**2+qz**2),   2*(qy*qz-qx*qw)],
        [  2*(qx*qz-qy*qw),   2*(qy*qz+qx*qw), 1-2*(qx**2+qy**2)],
    ])

def colmap_txt_to_transforms(txt_dir, image_dir, out_path):
    focal = None
    with open(txt_dir / "cameras.txt", encoding="utf-8") as f:
        for line in f:
            if line.startswith("#") or not line.strip():
                continue
            parts = line.split()
            model = parts[1]
            W, H  = int(parts[2]), int(parts[3])
            focal = float(parts[4])
            break
    if focal is None:
        raise RuntimeError("無法從 cameras.txt 解析 focal length")

    camera_angle_x = 2 * np.arctan(W / (2 * focal))
    frames = []
    with open(txt_dir / "images.txt", encoding="utf-8") as f:
        lines = [l for l in f if not l.startswith("#") and l.strip()]
    i = 0
    while i < len(lines):
        parts           = lines[i].split()
        qw, qx, qy, qz = map(float, parts[1:5])
        tx, ty, tz      = map(float, parts[5:8])
        fname           = parts[9]
        i += 2
        R   = quat_to_rot(qw, qx, qy, qz)
        t   = np.array([tx, ty, tz])
        c2w = np.eye(4)
        c2w[:3, :3] = R.T
        c2w[:3,  3] = -R.T @ t
        c2w[:, 1]  *= -1
        c2w[:, 2]  *= -1
        # 儲存為正斜線路徑，跨平台相容
        frames.append({"file_path": str(image_dir / fname).replace("\\", "/"),
                       "transform_matrix": c2w.tolist()})

    transforms = {"camera_angle_x": camera_angle_x, "frames": frames}
    out_path.write_text(json.dumps(transforms, indent=2), encoding="utf-8")
    return transforms

def run_colmap(image_dir, output_dir):
    ws     = output_dir / "colmap_ws"
    db     = ws / "database.db"
    sparse = ws / "sparse"
    ws.mkdir(parents=True, exist_ok=True)
    sparse.mkdir(exist_ok=True)
    c      = str(COLMAP_BIN)

    for cmd in [
        [c, "feature_extractor",
         "--database_path", str(db),
         "--image_path",    str(image_dir),
         "--ImageReader.single_camera",       "1",
         "--SiftExtraction.max_num_features", "16384"],
        [c, "exhaustive_matcher",
         "--database_path", str(db),
         "--TwoViewGeometry.min_num_inliers", "10"],
        [c, "mapper",
         "--database_path", str(db),
         "--image_path",    str(image_dir),
         "--output_path",   str(sparse)],
    ]:
        print(f"  → {cmd[1]} ...")
        run_cmd(cmd)

    recon_dirs = sorted(sparse.iterdir())
    if not recon_dirs:
        raise RuntimeError("COLMAP mapper 未產生任何 reconstruction，照片重疊不足")

    txt_dir = output_dir / "colmap_text"
    txt_dir.mkdir(parents=True, exist_ok=True)
    run_cmd([c, "model_converter",
             "--input_path",  str(recon_dirs[0]),
             "--output_path", str(txt_dir),
             "--output_type", "TXT"])

    return colmap_txt_to_transforms(txt_dir, image_dir, output_dir / "transforms.json")

# 執行（有快取則跳過）
COLMAP_OUT.mkdir(parents=True, exist_ok=True)
cache = COLMAP_OUT / "transforms.json"

if cache.exists():
    n_reg = len(json.loads(cache.read_text(encoding="utf-8"))["frames"])
    print(f"[快取] transforms.json 已存在，{n_reg} frames")
else:
    imgs = list({p.name: p for ext in ("*.jpg", "*.jpeg", "*.png")
                 for p in PHOTO_DIR.glob(ext)}.values())
    print(f"[COLMAP] {len(imgs)} 張照片 ...")
    transforms = run_colmap(PHOTO_DIR, COLMAP_OUT)
    print(f"✅ 成功註冊 {len(transforms['frames'])} 張")

[快取] transforms.json 已存在，30 frames


In [5]:
class NeRF(nn.Module):
    def __init__(self, pos_dim, dir_dim, hidden=256):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Linear(pos_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),  nn.ReLU(),
            nn.Linear(hidden, hidden),  nn.ReLU(),
            nn.Linear(hidden, hidden),  nn.ReLU(),
        )
        self.block2 = nn.Sequential(
            nn.Linear(hidden + pos_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),           nn.ReLU(),
            nn.Linear(hidden, hidden),           nn.ReLU(),
            nn.Linear(hidden, hidden + 1),
        )
        self.color_head = nn.Sequential(
            nn.Linear(hidden + dir_dim, hidden // 2), nn.ReLU(),
            nn.Linear(hidden // 2, 3),                nn.Sigmoid(),
        )

    def forward(self, pos_enc, dir_enc):
        h     = self.block1(pos_enc)
        h     = self.block2(torch.cat([h, pos_enc], dim=-1))
        sigma = torch.relu(h[..., 0])
        rgb   = self.color_head(torch.cat([h[..., 1:], dir_enc], dim=-1))
        return rgb, sigma

def positional_encoding(x, num_freqs):
    freqs = 2.0 ** torch.linspace(0, num_freqs - 1, num_freqs, device=x.device)
    return torch.cat([x] + [fn(f * x) for f in freqs for fn in (torch.sin, torch.cos)], dim=-1)

def get_rays(H, W, focal, c2w):
    i, j = torch.meshgrid(
        torch.arange(W, dtype=torch.float32, device=c2w.device),
        torch.arange(H, dtype=torch.float32, device=c2w.device),
        indexing="xy",
    )
    dirs   = torch.stack([(i-W*0.5)/focal, -(j-H*0.5)/focal, -torch.ones_like(i)], dim=-1)
    rays_d = (dirs[..., None, :] * c2w[:3, :3]).sum(-1)
    rays_o = c2w[:3, 3].expand(rays_d.shape)
    return rays_o, rays_d

def sample_points(rays_o, rays_d, near, far, n_samples, perturb=False):
    t_vals = torch.linspace(0, 1, n_samples, device=rays_o.device)
    z_vals = (near*(1-t_vals) + far*t_vals).expand(rays_o.shape[0], n_samples)
    if perturb:
        mids   = 0.5 * (z_vals[..., 1:] + z_vals[..., :-1])
        upper  = torch.cat([mids, z_vals[..., -1:]], dim=-1)
        lower  = torch.cat([z_vals[..., :1], mids],  dim=-1)
        z_vals = lower + (upper - lower) * torch.rand_like(z_vals)
    return rays_o[..., None, :] + rays_d[..., None, :] * z_vals[..., :, None], z_vals

def volume_render(rgb, sigma, z_vals, bg_color=1.0):
    deltas  = torch.cat([z_vals[..., 1:] - z_vals[..., :-1],
                         torch.full_like(z_vals[..., :1], 1e10)], dim=-1)
    alpha   = 1.0 - torch.exp(-sigma * deltas)
    T       = torch.cumprod(torch.cat([torch.ones_like(alpha[..., :1]),
                                       1-alpha+1e-10], dim=-1), dim=-1)[..., :-1]
    weights = T * alpha
    rgb_map = (weights[..., None] * rgb).sum(-2) + (1 - weights.sum(-1, keepdim=True)) * bg_color
    return rgb_map, weights

def render_pose(model, pose, H, W, focal):
    model.eval()
    rays_o, rays_d = get_rays(H, W, focal, pose.to(device))
    rays_o, rays_d = rays_o.reshape(-1, 3), rays_d.reshape(-1, 3)
    chunks = []
    with torch.no_grad():
        for k in range(0, rays_o.shape[0], BATCH_RAY):
            ro, rd  = rays_o[k:k+BATCH_RAY], rays_d[k:k+BATCH_RAY]
            pts, zv = sample_points(ro, rd, NEAR, FAR, N_SAMPLES)
            pe      = positional_encoding(pts.reshape(-1, 3), POS_FREQS)
            de      = positional_encoding(rd[:, None].expand_as(pts).reshape(-1, 3), DIR_FREQS)
            rgb_raw, sigma_raw = model(pe, de)
            rgb_map, _ = volume_render(rgb_raw.reshape(-1, N_SAMPLES, 3),
                                       sigma_raw.reshape(-1, N_SAMPLES), zv)
            chunks.append(rgb_map.cpu())
    return torch.cat(chunks).reshape(H, W, 3).numpy().clip(0, 1)

def estimate_near_far(txt_dir, poses, percentile=5):
    pts3d_path = txt_dir / "points3D.txt"
    pts = []
    with open(pts3d_path, encoding="utf-8") as f:
        for line in f:
            if line.startswith("#") or not line.strip():
                continue
            parts = line.split()
            pts.append([float(parts[1]), float(parts[2]), float(parts[3])])

    if not pts:
        print("  [警告] 點雲為空，從相機位置估算 NEAR/FAR")
        cam_origins = np.stack([pose.numpy()[:3, 3] for pose in poses])
        dists = np.linalg.norm(cam_origins - cam_origins.mean(axis=0), axis=1)
        radius = float(dists.max()) + 0.5
        near, far = max(radius * 0.1, 0.05), radius * 3.0
        print(f"  NEAR={near:.3f}  FAR={far:.3f}  (from camera origins)")
        return near, far

    pts    = np.array(pts)
    depths = []
    for pose in poses:
        cam_origin = pose.numpy()[:3, 3]
        dists      = np.linalg.norm(pts - cam_origin, axis=1)
        depths.extend(dists.tolist())

    if not depths:
        print("  [警告] 無正深度點，使用預設 NEAR/FAR=0.1/10.0")
        return 0.1, 10.0

    depths = np.array(depths)
    near   = max(float(np.percentile(depths, percentile)) * 0.8, 1e-3)
    far    = float(np.percentile(depths, 100 - percentile)) * 1.2
    print(f"  NEAR={near:.3f}  FAR={far:.3f}  (from {len(pts)} 3D points)")
    return near, far

def load_dataset(test_ratio=0.2):
    meta   = json.loads((COLMAP_OUT / "transforms.json").read_text(encoding="utf-8"))
    frames = meta["frames"]
    n_test = max(1, int(len(frames) * test_ratio))

    def load_frames(flist):
        imgs, poses = [], []
        for frame in flist:
            img = imageio.imread(frame["file_path"]).astype(np.float32) / 255.0
            if img.ndim == 2:
                img = np.stack([img]*3, axis=-1)
            if img.shape[-1] == 4:
                img = img[..., :3] * img[..., 3:] + (1 - img[..., 3:])
            H0, W0 = img.shape[:2]
            img = np.array(PILImage.fromarray((img*255).astype(np.uint8)).resize(
                (int(W0*SCALE), int(H0*SCALE)), PILImage.LANCZOS)) / 255.0
            imgs.append(img)
            poses.append(np.array(frame["transform_matrix"], dtype=np.float32))
        return imgs, poses

    train_imgs, train_poses = load_frames(frames[:-n_test])
    test_imgs,  test_poses  = load_frames(frames[-n_test:])
    H, W  = train_imgs[0].shape[:2]
    focal = 0.5 * int(W/SCALE) / np.tan(0.5 * meta["camera_angle_x"]) * SCALE
    tt    = lambda lst: torch.tensor(np.stack(lst), dtype=torch.float32)

    all_poses = tt(train_imgs + test_imgs)   # dummy, just need pose tensors
    all_poses = torch.tensor(
        np.stack([np.array(f["transform_matrix"], dtype=np.float32) for f in frames])
    )
    near, far = estimate_near_far(COLMAP_OUT / "colmap_text", all_poses)
    return tt(train_imgs), tt(train_poses), tt(test_imgs), tt(test_poses), H, W, focal, near, far

In [ ]:
train_imgs, train_poses, test_imgs, test_poses, H, W, focal, NEAR, FAR = load_dataset()
n_train = len(train_imgs)
print(f"Train: {n_train} 張  |  Test: {len(test_imgs)} 張  |  {H}×{W}  focal={focal:.1f}")
print(f"NEAR={NEAR:.3f}  FAR={FAR:.3f}")

origins   = train_poses[:, :3, 3].numpy()
mean_dist = float(np.linalg.norm(origins - origins.mean(0), axis=1).mean())
print(f"Camera X:[{origins[:,0].min():.3f},{origins[:,0].max():.3f}]  "
      f"Y:[{origins[:,1].min():.3f},{origins[:,1].max():.3f}]  "
      f"Z:[{origins[:,2].min():.3f},{origins[:,2].max():.3f}]")
print(f"mean_dist={mean_dist:.3f}  NEAR={NEAR:.3f}  FAR={FAR:.3f}  (建議 FAR > {mean_dist*2:.2f})")

np.random.seed(42); torch.manual_seed(42)
model     = NeRF(pos_dim, dir_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.1**(1/N_ITERS))
losses, t0 = [], time.time()

for _ in tqdm(range(N_ITERS), desc="Training"):
    idx             = np.random.randint(n_train)
    img, pose       = train_imgs[idx].to(device), train_poses[idx].to(device)
    rays_o, rays_d  = get_rays(H, W, focal, pose)
    rays_o, rays_d  = rays_o.reshape(-1, 3), rays_d.reshape(-1, 3)
    target          = img.reshape(-1, 3)
    sel             = np.random.choice(rays_o.shape[0], BATCH_RAY, replace=False)
    rays_o, rays_d, target = rays_o[sel], rays_d[sel], target[sel]
    pts, z_vals     = sample_points(rays_o, rays_d, NEAR, FAR, N_SAMPLES, perturb=True)
    pe              = positional_encoding(pts.reshape(-1, 3), POS_FREQS)
    de              = positional_encoding(rays_d[:, None].expand_as(pts).reshape(-1, 3), DIR_FREQS)
    rgb_raw, sigma_raw = model(pe, de)
    rgb_map, _      = volume_render(rgb_raw.reshape(BATCH_RAY, N_SAMPLES, 3),
                                    sigma_raw.reshape(BATCH_RAY, N_SAMPLES), z_vals)
    loss = ((rgb_map - target)**2).mean()
    optimizer.zero_grad(); loss.backward(); optimizer.step(); scheduler.step()
    losses.append(loss.item())

torch.save(model.state_dict(), "ckpt_real.pt")
print(f"訓練完成  {(time.time()-t0)/60:.1f} 分鐘")

  NEAR=9.222  FAR=29.639  (from 2862 3D points)
Train: 24 張  |  Test: 6 張  |  1734×2312  focal=1832.3
NEAR=9.222  FAR=29.639
Camera X:[-0.241,0.748]  Y:[-6.059,2.299]  Z:[-1.470,1.114]
mean_dist=1.667  NEAR=9.222  FAR=29.639  (建議 FAR > 3.33)


Training:   0%|          | 54/15000 [00:24<1:44:37,  2.38it/s]

In [ ]:
# 若從新 kernel 執行，先載入 checkpoint
# model = NeRF(pos_dim, dir_dim).to(device)
# model.load_state_dict(torch.load("ckpt_real.pt", map_location=device))

model.eval()
psnrs, ssims, renders = [], [], []
for i in range(len(test_imgs)):
    gt   = test_imgs[i].numpy()
    pred = render_pose(model, test_poses[i], H, W, focal)
    renders.append(pred)
    psnrs.append(psnr_fn(gt, pred, data_range=1.0))
    ssims.append(ssim_fn(gt, pred, data_range=1.0, channel_axis=-1))

mean_psnr, mean_ssim = float(np.mean(psnrs)), float(np.mean(ssims))
print(f"PSNR: {mean_psnr:.2f} dB  |  SSIM: {mean_ssim:.4f}")

# ── Loss 曲線 ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
sm = np.convolve(losses, np.ones(200)/200, mode="valid")
ax.plot(sm, color="#2563EB", linewidth=1.8)
ax.set(xlabel="Iteration", ylabel="MSE Loss (smoothed)", title="Training Loss — Real Photo NeRF")
ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig("loss_curve_real.png", dpi=150, bbox_inches="tight"); plt.show()

# ── 視覺對比（GT vs Rendered）─────────────────────────────
n_show = min(len(test_imgs), 4)
fig, axes = plt.subplots(2, n_show, figsize=(5*n_show, 9), squeeze=False)
for col in range(n_show):
    gt, pred = test_imgs[col].numpy(), renders[col]
    axes[0, col].imshow(gt.clip(0, 1))
    axes[0, col].set_title(f"GT #{col}", fontsize=10)
    axes[0, col].axis("off")
    p = psnr_fn(gt, pred, data_range=1.0)
    s = ssim_fn(gt, pred, data_range=1.0, channel_axis=-1)
    axes[1, col].imshow(pred.clip(0, 1))
    axes[1, col].set_title(f"Rendered\nPSNR={p:.1f} | SSIM={s:.3f}", fontsize=9)
    axes[1, col].axis("off")
plt.suptitle(f"Real Photo NeRF — mean PSNR={mean_psnr:.2f} dB | SSIM={mean_ssim:.4f}",
             fontsize=13, fontweight="bold")
plt.tight_layout(); plt.savefig("visual_comparison_real.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# 若從新 kernel 執行，先載入 checkpoint
# model = NeRF(pos_dim, dir_dim).to(device)
# model.load_state_dict(torch.load("ckpt_real.pt", map_location=device))

def generate_360_poses(train_poses, n_frames=120):
    origins = train_poses[:, :3, 3].numpy()
    center  = origins.mean(axis=0)
    radius  = float(np.linalg.norm(origins - center, axis=1).mean())
    height  = float(origins[:, 1].mean())

    frames = []
    for i in range(n_frames):
        angle = 2 * np.pi * i / n_frames
        cam_pos = np.array([
            center[0] + radius * np.cos(angle),
            height,
            center[2] + radius * np.sin(angle),
        ], dtype=np.float32)
        forward = center.astype(np.float32) - cam_pos
        forward /= np.linalg.norm(forward)
        up      = np.array([0, -1, 0], dtype=np.float32)
        right   = np.cross(forward, up)
        right  /= np.linalg.norm(right)
        up      = np.cross(right, forward)
        c2w = np.eye(4, dtype=np.float32)
        c2w[:3, 0] = right
        c2w[:3, 1] = up
        c2w[:3, 2] = -forward
        c2w[:3, 3] = cam_pos
        frames.append(torch.tensor(c2w))
    return frames

video_poses  = generate_360_poses(train_poses, n_frames=120)
video_frames = []
for pose in tqdm(video_poses, desc="Rendering 360°"):
    frame = render_pose(model, pose, H, W, focal)
    video_frames.append((frame * 255).astype(np.uint8))

imageio.mimwrite("nerf_360_preview.mp4", video_frames, fps=30, quality=8)
print("✅ 已儲存 nerf_360_preview.mp4")